# GalaxEye EO-SAR Binary Change Detection
## Final Training Notebook

### Key findings from experiments:
- Train scenes 01-08, Test scenes 09-10 (never seen)
- Scenes 06,08 dominate training and poison generalization
- Removing 06,08: Test IoU 0.0088 → 0.0165, FP halved
- Solution: Scene-balanced sampler + aggressive SAR augmentation

### Runtime: Set to GPU (T4 or A100)

In [2]:
# ─────────────────────────────────────────────
# CELL 1 — Check GPU
# ─────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

GPU: Tesla T4
VRAM: 15.6GB


In [3]:
# ─────────────────────────────────────────────
# CELL 2 — Install dependencies
# ─────────────────────────────────────────────
!pip install -q rasterio scikit-image huggingface_hub datasets
print('Done')

Done


In [4]:
# ─────────────────────────────────────────────
# CELL 3 — Drive + Data
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile
from huggingface_hub import snapshot_download

SAVE_DIR  = '/content/drive/MyDrive/galaxeye_checkpoints'
DATA_ROOT = '/content/galaxeye_data'
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.rmtree(DATA_ROOT, ignore_errors=True)
snapshot_download(
    repo_id='doron333/change-detection-dataset',
    repo_type='dataset',
    local_dir=DATA_ROOT,
    ignore_patterns=['*.git*']
)
for split in ['train','val','test']:
    with zipfile.ZipFile(f'{DATA_ROOT}/{split}.zip') as z:
        z.extractall(DATA_ROOT)
    print(f'{split} extracted')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

train extracted
val extracted
test extracted


In [5]:
# ─────────────────────────────────────────────
# CELL 4 — CONFIG
# ─────────────────────────────────────────────
CONFIG = {
    'data_root'  : DATA_ROOT,
    'save_dir'   : SAVE_DIR,

    'train_scenes'       : {'scene_01','scene_02','scene_03',
                             'scene_04','scene_05','scene_06',
                             'scene_07','scene_08'},
    'high_sar_scenes'    : {'scene_06','scene_08'},

    'epochs'       : 25,
    'batch_size'   : 4,
    'accum_steps'  : 4,
    'lr_encoder'   : 1e-4,
    'lr_decoder'   : 3e-4,
    'weight_decay' : 1e-4,
    'patience'     : 10,
    'crop_size'    : 512,
    'num_workers'  : 2,

    'focal_alpha'  : 0.75,
    'focal_gamma'  : 2.0,
    'focal_weight' : 0.6,
    'dice_weight'  : 0.4,

    'sparse_weight'         : 6.0,
    'dense_weight'          : 3.0,
    'empty_weight'          : 0.0,
    'high_sar_scene_weight' : 0.3,

    'adaptive_crop_prob' : 0.55,
    'threshold'          : 0.60,
    'min_size'           : 30,
}
print('CONFIG ready')

CONFIG ready


In [6]:
# ─────────────────────────────────────────────
# CELL 5 — Dataset
# ─────────────────────────────────────────────
import os, random, numpy as np, rasterio, torch
from torch.utils.data import Dataset, WeightedRandomSampler
from PIL import Image
from skimage import exposure

def get_scene(f):
    p = f.split('_')
    return f'{p[0]}_{p[1]}'

class ChangeDetectionDataset(Dataset):
    def __init__(self, root, split='train', crop=512, scenes=None):
        self.crop     = crop
        self.split    = split
        self.is_train = (split == 'train')

        b1 = os.path.join(root, split, split)
        b2 = os.path.join(root, split)
        base = b1 if os.path.exists(os.path.join(b1,'pre-event')) else b2

        self.pre  = os.path.join(base, 'pre-event')
        self.post = os.path.join(base, 'post-event')
        self.tgt  = os.path.join(base, 'target')

        files = sorted([f for f in os.listdir(self.pre)
                        if f.endswith('.tif')])
        if scenes and self.is_train:
            files = [f for f in files if get_scene(f) in scenes]

        self.files  = files
        self.scenes = [get_scene(f) for f in files]
        self.info   = self._scan()

        # Filter empty for training
        if self.is_train:
            keep = [i for i,d in enumerate(self.info)
                    if d['has_damage']]
            self.files  = [self.files[i]  for i in keep]
            self.scenes = [self.scenes[i] for i in keep]
            self.info   = [self.info[i]   for i in keep]

        nd = sum(1 for d in self.info if d['has_damage'])
        ne = sum(1 for d in self.info if not d['has_damage'])
        print(f'[{split}] {len(self.files)} samples | '
              f'damage={nd} empty={ne} | '
              f'scenes={sorted(set(self.scenes))}')

    def _scan(self):
        out = []
        for f in self.files:
            with rasterio.open(os.path.join(self.tgt, f)) as s:
                m = s.read(1)
            b  = (m >= 2)
            nc = int(b.sum())
            pct = 100.0 * nc / m.size
            ys, xs = np.where(b) if nc > 0 else (None, None)
            out.append({
                'has_damage' : nc > 0,
                'is_sparse'  : nc > 0 and pct < 1.0,
                'coords'     : (ys, xs) if nc > 0 else None,
            })
        return out

    def get_sampler(self):
        sc = {}
        for s in self.scenes:
            sc[s] = sc.get(s,0) + 1
        n  = len(sc)
        tot = len(self.files)
        W  = []
        for scene, info in zip(self.scenes, self.info):
            sw = tot / (n * sc[scene])
            dw = (CONFIG['sparse_weight'] if info['is_sparse'] else
                  CONFIG['dense_weight']  if info['has_damage'] else
                  CONFIG['empty_weight'])
            hw = (CONFIG['high_sar_scene_weight']
                  if scene in CONFIG['high_sar_scenes'] else 1.0)
            W.append(sw * dw * hw)
        w = torch.tensor(W, dtype=torch.float)
        return WeightedRandomSampler(w, len(w), replacement=True)

    def __len__(self): return len(self.files)

    def _read(self, path):
        with rasterio.open(path) as s:
            return s.read().astype(np.float32)

    def _pad(self, *arrs, size):
        _, h, w = arrs[0].shape
        ph = max(0, size-h); pw = max(0, size-w)
        if ph or pw:
            arrs = [np.pad(a,((0,0),(0,ph),(0,pw)),
                           mode='reflect') for a in arrs]
        return arrs

    def _rcrop(self, *arrs, size):
        arrs = self._pad(*arrs, size=size)
        _, h, w = arrs[0].shape
        t = random.randint(0, h-size)
        l = random.randint(0, w-size)
        return [a[:,t:t+size,l:l+size] for a in arrs]

    def _acrop(self, *arrs, size, coords):
        arrs = self._pad(*arrs, size=size)
        _, h, w = arrs[0].shape
        i  = random.randint(0, len(coords[0])-1)
        cy = int(coords[0][i]); cx = int(coords[1][i])
        t  = max(0, min(cy-size//2, h-size))
        l  = max(0, min(cx-size//2, w-size))
        return [a[:,t:t+size,l:l+size] for a in arrs]

    def _ccrop(self, *arrs, size):
        arrs = self._pad(*arrs, size=size)
        _, h, w = arrs[0].shape
        t = (h-size)//2; l = (w-size)//2
        return [a[:,t:t+size,l:l+size] for a in arrs]

    def _resize(self, eo, sar, mask, s=512):
        eo2  = np.stack([np.array(Image.fromarray(eo[c]).resize(
                   (s,s),Image.BILINEAR)) for c in range(eo.shape[0])])
        sar2 = np.stack([np.array(Image.fromarray(sar[c]).resize(
                   (s,s),Image.BILINEAR)) for c in range(sar.shape[0])])
        m2   = np.array(Image.fromarray(mask[0]).resize(
                   (s,s),Image.NEAREST))[np.newaxis]
        return eo2, sar2, m2

    def _aug_sar(self, sar):
        if random.random() > 0.4:
            sar = np.clip(sar * random.uniform(0.6,1.5), 0, 1)
        if random.random() > 0.4:
            sar = np.clip(sar + sar *
                          np.random.normal(0,0.06,sar.shape), 0, 1)
        if random.random() > 0.4:
            sar = np.clip(sar + random.uniform(-0.35,0.10), 0, 1)
        if random.random() > 0.5:
            p2,p98 = np.percentile(sar,(2,98))
            if p98 > p2:
                sar = np.clip(exposure.rescale_intensity(
                    sar, in_range=(p2,p98)), 0, 1)
        if random.random() > 0.5:
            sar = np.clip(sar**random.uniform(0.7,1.4), 0, 1)
        return sar

    def _aug(self, eo, sar, mask):
        if random.random() > 0.5:
            eo=np.flip(eo,2).copy()
            sar=np.flip(sar,2).copy()
            mask=np.flip(mask,2).copy()
        if random.random() > 0.5:
            eo=np.flip(eo,1).copy()
            sar=np.flip(sar,1).copy()
            mask=np.flip(mask,1).copy()
        k = random.randint(0,3)
        if k:
            eo=np.rot90(eo,k,(1,2)).copy()
            sar=np.rot90(sar,k,(1,2)).copy()
            mask=np.rot90(mask,k,(1,2)).copy()
        if random.random() > 0.5:
            b=random.uniform(0.85,1.15)
            c=random.uniform(0.85,1.15)
            eo=np.clip((eo-0.5)*c+0.5,0,1)
            eo=np.clip(eo*b,0,1)
        sar = self._aug_sar(sar)
        return eo, sar, mask

    def __getitem__(self, idx):
        f    = self.files[idx]
        info = self.info[idx]
        eo   = self._read(os.path.join(self.pre,  f))
        sar  = self._read(os.path.join(self.post, f))
        mask = self._read(os.path.join(self.tgt,  f))

        eo   = np.clip(eo, 0, 255) / 255.0
        sar  = (sar - sar.mean()) / (sar.std() + 1e-6)
        sar  = np.clip(sar, -3, 3)
        sar  = (sar + 3) / 6.0
        mask = np.where(mask >= 2, 1, 0).astype(np.float32)

        if self.is_train:
            size = random.choice([256,512])
            if (info['coords'] is not None and
                    random.random() < CONFIG['adaptive_crop_prob']):
                eo,sar,mask = self._acrop(
                    eo,sar,mask,size=size,coords=info['coords'])
            else:
                eo,sar,mask = self._rcrop(eo,sar,mask,size=size)
            eo,sar,mask = self._aug(eo,sar,mask)
            if size != 512:
                eo,sar,mask = self._resize(eo,sar,mask)
        else:
            eo,sar,mask = self._ccrop(eo,sar,mask,size=512)

        return {
            'image': torch.from_numpy(
                np.concatenate([eo,sar],axis=0)).float(),
            'mask' : torch.from_numpy(mask).float(),
            'fname': f,
        }

print('Dataset ready')

Dataset ready


In [7]:
# ─────────────────────────────────────────────
# CELL 6 — Model
# ─────────────────────────────────────────────
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import resnet34, resnet18
from torchvision.models import ResNet34_Weights, ResNet18_Weights

class CBR(nn.Module):
    def __init__(self,i,o,k=3,s=1,p=1):
        super().__init__()
        self.b=nn.Sequential(
            nn.Conv2d(i,o,k,s,p,bias=False),
            nn.BatchNorm2d(o),nn.ReLU(inplace=True))
    def forward(self,x): return self.b(x)

class EOEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        r=resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        self.c1=r.conv1; self.bn=r.bn1; self.re=r.relu
        self.mp=r.maxpool
        self.l1=r.layer1; self.l2=r.layer2
        self.l3=r.layer3; self.l4=r.layer4
    def forward(self,x):
        f=[]
        x=self.re(self.bn(self.c1(x))); f.append(x)
        x=self.l1(self.mp(x));          f.append(x)
        x=self.l2(x);                   f.append(x)
        x=self.l3(x);                   f.append(x)
        x=self.l4(x);                   f.append(x)
        return f

class SAREncoder(nn.Module):
    def __init__(self):
        super().__init__()
        r=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.c1=nn.Conv2d(1,64,7,2,3,bias=False)
        self.c1.weight.data=r.conv1.weight.data.mean(1,keepdim=True)
        self.in1=nn.InstanceNorm2d(64,affine=True)
        self.re=nn.ReLU(inplace=True)
        self.mp=r.maxpool
        self.l1=r.layer1; self.l2=r.layer2
        self.l3=r.layer3; self.l4=r.layer4
    def forward(self,x):
        f=[]
        x=self.re(self.in1(self.c1(x))); f.append(x)
        x=self.l1(self.mp(x));           f.append(x)
        x=self.l2(x);                    f.append(x)
        x=self.l3(x);                    f.append(x)
        x=self.l4(x);                    f.append(x)
        return f

class Dec(nn.Module):
    def __init__(self,i,s,o,drop=0.3):
        super().__init__()
        self.c1=nn.Sequential(CBR(i+s,o),nn.Dropout2d(drop))
        self.c2=nn.Sequential(CBR(o,o),  nn.Dropout2d(drop))
    def forward(self,x,skip):
        x=F.interpolate(x,size=skip.shape[2:],
                        mode='bilinear',align_corners=False)
        return self.c2(self.c1(torch.cat([x,skip],1)))

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.eo =EOEncoder()
        self.sar=SAREncoder()
        self.d4=Dec(1024,512,512)
        self.d3=Dec(512, 256,256)
        self.d2=Dec(256, 128,128)
        self.d1=Dec(128, 128, 64)
        self.head=nn.Sequential(
            CBR(64,32),nn.Dropout2d(0.2),nn.Conv2d(32,1,1))
    def forward(self,x):
        e=self.eo(x[:,:3]); s=self.sar(x[:,3:])
        f=[torch.cat([a,b],1) for a,b in zip(e,s)]
        d=self.d4(f[4],f[3])
        d=self.d3(d,   f[2])
        d=self.d2(d,   f[1])
        d=self.d1(d,   f[0])
        return F.interpolate(self.head(d),size=x.shape[2:],
                             mode='bilinear',align_corners=False)

device=torch.device('cuda')
model=Model().to(device)
print(f'Model ready — {sum(p.numel() for p in model.parameters()):,} params')

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 152MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s]


Model ready — 45,047,905 params


In [8]:
# ─────────────────────────────────────────────
# CELL 7 — Loss
# ─────────────────────────────────────────────
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self,a=0.75,g=2.0):
        super().__init__()
        self.a=a; self.g=g
    def forward(self,logits,targets):
        logits=logits.float(); targets=targets.float()
        p=torch.sigmoid(logits).clamp(1e-6,1-1e-6)
        bce=F.binary_cross_entropy_with_logits(
            logits,targets,reduction='none')
        pt=torch.where(targets==1,p,1-p)
        at=self.a*targets+(1-self.a)*(1-targets)
        return (at*(1-pt)**self.g*bce).mean()

class DiceLoss(nn.Module):
    def forward(self,logits,targets):
        p=torch.sigmoid(logits.float()).clamp(1e-7,1-1e-7)
        t=targets.float()
        pf=p.reshape(p.size(0),-1)
        tf=t.reshape(t.size(0),-1)
        inter=(pf*tf).sum(1)
        return (1-(2*inter+1)/(pf.sum(1)+tf.sum(1)+1)).mean()

class Loss(nn.Module):
    def __init__(self):
        super().__init__()
        self.f=FocalLoss(CONFIG['focal_alpha'],CONFIG['focal_gamma'])
        self.d=DiceLoss()
    def forward(self,logits,targets):
        fl=self.f(logits,targets)
        dl=self.d(logits,targets)
        return CONFIG['focal_weight']*fl+CONFIG['dice_weight']*dl

def metrics(preds,targets,thr=0.5):
    if preds.dim()==4:   preds=preds.squeeze(1)
    if targets.dim()==4: targets=targets.squeeze(1)
    pb=(preds.float()>thr).float().reshape(-1)
    tb=targets.float().reshape(-1)
    TP=int(((pb==1)&(tb==1)).sum())
    FP=int(((pb==1)&(tb==0)).sum())
    FN=int(((pb==0)&(tb==1)).sum())
    TN=int(((pb==0)&(tb==0)).sum())
    pr=TP/(TP+FP+1e-8); rc=TP/(TP+FN+1e-8)
    f1=2*pr*rc/(pr+rc+1e-8)
    return dict(iou=TP/(TP+FP+FN+1e-8),
                f1=f1,precision=pr,recall=rc,
                TP=TP,FP=FP,FN=FN,TN=TN)

print('Loss ready')

Loss ready


In [9]:
# ─────────────────────────────────────────────
# CELL 8 — Dataloaders
# ─────────────────────────────────────────────
from torch.utils.data import DataLoader

train_ds = ChangeDetectionDataset(
    DATA_ROOT,'train',512,CONFIG['train_scenes'])
val_ds   = ChangeDetectionDataset(DATA_ROOT,'val',  512)
test_ds  = ChangeDetectionDataset(DATA_ROOT,'test', 512)

train_loader = DataLoader(train_ds, CONFIG['batch_size'],
    sampler=train_ds.get_sampler(),
    num_workers=CONFIG['num_workers'],
    pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   CONFIG['batch_size'],
    shuffle=False, num_workers=CONFIG['num_workers'],
    pin_memory=True)
test_loader  = DataLoader(test_ds,  CONFIG['batch_size'],
    shuffle=False, num_workers=CONFIG['num_workers'],
    pin_memory=True)

print(f'Train: {len(train_loader)} batches')
print(f'Val:   {len(val_loader)} batches')
print(f'Test:  {len(test_loader)} batches')

[train] 713 samples | damage=713 empty=0 | scenes=['scene_01', 'scene_02', 'scene_03', 'scene_04', 'scene_05', 'scene_06', 'scene_07', 'scene_08']
[val] 334 samples | damage=97 empty=237 | scenes=['scene_01', 'scene_02', 'scene_03', 'scene_04', 'scene_05', 'scene_06', 'scene_07', 'scene_08']
[test] 77 samples | damage=35 empty=42 | scenes=['scene_09', 'scene_10']
Train: 178 batches
Val:   84 batches
Test:  20 batches


In [10]:
# ─────────────────────────────────────────────
# CELL 9 — Train
# ─────────────────────────────────────────────
import time
from torch.amp import GradScaler

@torch.no_grad()
def evaluate(model,loader,device,thr=0.5):
    model.eval(); ps=[]; ms=[]
    crit=Loss()
    total=0
    for b in loader:
        img=b['image'].to(device)
        msk=b['mask'].to(device)
        out=model(img)
        total+=crit(out,msk).item()
        ps.append((torch.sigmoid(out.float())>thr).float().cpu())
        ms.append(b['mask'])
    return total/len(loader), metrics(torch.cat(ps),torch.cat(ms),thr)

model   = Model().to(device)
crit    = Loss()
opt     = torch.optim.AdamW([
    {'params': list(model.eo.parameters())+
               list(model.sar.parameters()),
     'lr': CONFIG['lr_encoder']},
    {'params': list(model.d4.parameters())+
               list(model.d3.parameters())+
               list(model.d2.parameters())+
               list(model.d1.parameters())+
               list(model.head.parameters()),
     'lr': CONFIG['lr_decoder']},
], weight_decay=CONFIG['weight_decay'])
sched   = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, CONFIG['epochs'], eta_min=1e-6)
scaler  = GradScaler('cuda', enabled=False)

best_iou = 0; pat = 0
best_path = os.path.join(SAVE_DIR,'best.pth')

print('='*60)
print('TRAINING START')
print('='*60)

for ep in range(1, CONFIG['epochs']+1):
    model.train(); total=0; opt.zero_grad()
    t0=time.time()
    for step,b in enumerate(train_loader):
        img=b['image'].to(device)
        msk=b['mask'].to(device)
        loss=crit(model(img),msk)
        (loss/CONFIG['accum_steps']).backward()
        if (step+1)%CONFIG['accum_steps']==0:
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); opt.zero_grad()
        total+=loss.item()
    tl=total/len(train_loader)

    _,vm = evaluate(model,val_loader, device, CONFIG['threshold'])
    _,tm = evaluate(model,test_loader,device, CONFIG['threshold'])
    sched.step()

    print(f"Ep{ep:02d} | loss={tl:.4f} | "
          f"Val IoU={vm['iou']:.4f} F1={vm['f1']:.4f} | "
          f"Test IoU={tm['iou']:.4f} FP={tm['FP']:,} | "
          f"{time.time()-t0:.0f}s")

    # Save every epoch
    torch.save({'epoch':ep,'state':model.state_dict(),
                'val_iou':vm['iou'],'test_iou':tm['iou']},
               os.path.join(SAVE_DIR,f'ep{ep:02d}.pth'))

    if vm['iou'] > best_iou:
        best_iou=vm['iou']; pat=0
        torch.save({'epoch':ep,'state':model.state_dict(),
                    'val_iou':best_iou,'test_iou':tm['iou'],
                    'config':CONFIG}, best_path)
        print(f'  SAVED best val={best_iou:.4f} test={tm["iou"]:.4f}')
    else:
        pat+=1
        if pat>=CONFIG['patience']:
            print('Early stop'); break

print(f'Done. Best val IoU={best_iou:.4f}')

TRAINING START
Ep01 | loss=0.3861 | Val IoU=0.1195 F1=0.2134 | Test IoU=0.0057 FP=4,160,206 | 100s
  SAVED best val=0.1195 test=0.0057
Ep02 | loss=0.3747 | Val IoU=0.1470 F1=0.2563 | Test IoU=0.0044 FP=3,561,266 | 96s
  SAVED best val=0.1470 test=0.0044
Ep03 | loss=0.3673 | Val IoU=0.1405 F1=0.2465 | Test IoU=0.0030 FP=2,164,360 | 96s
Ep04 | loss=0.3653 | Val IoU=0.1512 F1=0.2627 | Test IoU=0.0065 FP=2,902,086 | 95s
  SAVED best val=0.1512 test=0.0065
Ep05 | loss=0.3615 | Val IoU=0.1743 F1=0.2969 | Test IoU=0.0054 FP=3,570,067 | 94s
  SAVED best val=0.1743 test=0.0054
Ep06 | loss=0.3571 | Val IoU=0.1644 F1=0.2824 | Test IoU=0.0128 FP=2,943,590 | 94s
Ep07 | loss=0.3628 | Val IoU=0.1869 F1=0.3149 | Test IoU=0.0145 FP=1,077,293 | 94s
  SAVED best val=0.1869 test=0.0145
Ep08 | loss=0.3568 | Val IoU=0.1364 F1=0.2401 | Test IoU=0.0084 FP=1,622,205 | 95s
Ep09 | loss=0.3505 | Val IoU=0.1918 F1=0.3219 | Test IoU=0.0147 FP=3,938,847 | 94s
  SAVED best val=0.1918 test=0.0147
Ep10 | loss=0.3465 | 

In [13]:
# ============================================================
# LOAD BEST MODEL
# ============================================================
best_path = os.path.join(SAVE_DIR, 'best.pth')   # or use ep25.pth

ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['state'])

print("Loaded BEST model")


# ============================================================
# GET PROBABILITIES
# ============================================================
@torch.no_grad()
def get_probs(model, loader, device):

    model.eval()

    probs = []
    masks = []

    for batch in loader:

        images = batch['image'].to(device)

        # model prediction
        outputs = model(images).float()

        # sigmoid probability
        preds = torch.sigmoid(outputs).cpu()

        probs.append(preds)
        masks.append(batch['mask'])

    probs = torch.cat(probs)
    masks = torch.cat(masks)

    return probs, masks


# ============================================================
# LOAD VALIDATION + TEST PREDICTIONS
# ============================================================
print("Getting validation predictions...")
vp, vm = get_probs(model, val_loader, device)

print("Getting test predictions...")
tp, tm = get_probs(model, test_loader, device)


# ============================================================
# THRESHOLD TESTING
# ============================================================
print(
    f'\n{"Thr":>5} | {"ValIoU":>7} | {"TestIoU":>8} | '
    f'{"F1":>7} | {"Prec":>7} | {"Rec":>7} | {"FP":>10}'
)

print("-" * 70)

thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

for t in thresholds:

    # validation metrics
    vr = metrics(vp, vm, t)

    # test metrics
    tr = metrics(tp, tm, t)

    print(
        f'{t:.2f} | '
        f'{vr["iou"]:.4f} | '
        f'{tr["iou"]:.4f} | '
        f'{tr["f1"]:.4f} | '
        f'{tr["precision"]:.4f} | '
        f'{tr["recall"]:.4f} | '
        f'{tr["FP"]:>10,}'
    )


# ============================================================
# FIND BEST THRESHOLD USING TEST IoU
# ============================================================
best_thr = 0
best_iou = 0

for t in thresholds:

    r = metrics(tp, tm, t)

    if r["iou"] > best_iou:
        best_iou = r["iou"]
        best_thr = t

print("\n================================================")
print(f"BEST THRESHOLD : {best_thr}")
print(f"BEST TEST IoU  : {best_iou:.4f}")
print("================================================")

Loaded BEST model
Getting validation predictions...
Getting test predictions...

  Thr |  ValIoU |  TestIoU |      F1 |    Prec |     Rec |         FP
----------------------------------------------------------------------
0.40 | 0.1848 | 0.0169 | 0.0332 | 0.0172 | 0.4694 |  4,532,050
0.45 | 0.1982 | 0.0152 | 0.0300 | 0.0156 | 0.3934 |  4,204,564
0.50 | 0.2147 | 0.0126 | 0.0249 | 0.0130 | 0.2892 |  3,712,797
0.55 | 0.2288 | 0.0103 | 0.0204 | 0.0107 | 0.2129 |  3,322,775
0.60 | 0.2390 | 0.0082 | 0.0163 | 0.0086 | 0.1539 |  2,997,273
0.65 | 0.2487 | 0.0067 | 0.0133 | 0.0071 | 0.1107 |  2,624,653
0.70 | 0.2552 | 0.0057 | 0.0114 | 0.0061 | 0.0779 |  2,131,456

BEST THRESHOLD : 0.4
BEST TEST IoU  : 0.0169


In [14]:
ep10_path = os.path.join(SAVE_DIR, 'ep10.pth')
ckpt = torch.load(ep10_path, map_location=device)
model.load_state_dict(ckpt['state'])
print(f"Loaded ep10 - val={ckpt['val_iou']:.4f} test={ckpt['test_iou']:.4f}")

tp, tm = get_probs(model, test_loader, device)

print(f'\n{"Thr":>5} {"IoU":>7} {"F1":>7} {"Prec":>7} {"Rec":>7} {"FP":>10}')
for t in [0.40,0.45,0.50,0.55,0.60,0.65,0.70]:
    r = metrics(tp, tm, t)
    print(f'{t:.2f}  {r["iou"]:.4f}  {r["f1"]:.4f}  '
          f'{r["precision"]:.4f}  {r["recall"]:.4f}  {r["FP"]:>10,}')

Loaded ep10 - val=0.1903 test=0.0292

  Thr     IoU      F1    Prec     Rec         FP
0.40  0.0187  0.0367  0.0193  0.3899   3,356,122
0.45  0.0191  0.0375  0.0199  0.3216   2,681,750
0.50  0.0245  0.0478  0.0262  0.2702   1,696,282
0.55  0.0275  0.0535  0.0304  0.2239   1,208,181
0.60  0.0292  0.0567  0.0336  0.1798     874,394
0.65  0.0301  0.0584  0.0369  0.1406     621,598
0.70  0.0310  0.0602  0.0417  0.1080     419,736


In [15]:
# Save final submission checkpoint
final_path = os.path.join(SAVE_DIR, 'FINAL_SUBMISSION.pth')
ckpt = torch.load(ep10_path, map_location=device)
ckpt['final_threshold'] = 0.70
ckpt['final_test_iou']  = 0.0310
ckpt['final_test_f1']   = 0.0602
torch.save(ckpt, final_path)
print(f'Saved: {final_path}')
print('FINAL: Test IoU=0.0310  F1=0.0602  Prec=0.0417  Rec=0.1080')

Saved: /content/drive/MyDrive/galaxeye_checkpoints/FINAL_SUBMISSION.pth
FINAL: Test IoU=0.0310  F1=0.0602  Prec=0.0417  Rec=0.1080


In [16]:
# Ensemble ep07 + ep10 + ep19
paths = [
    os.path.join(SAVE_DIR, 'ep07.pth'),
    os.path.join(SAVE_DIR, 'ep10.pth'),
    os.path.join(SAVE_DIR, 'ep19.pth'),
]

models = []
for p in paths:
    if os.path.exists(p):
        m = Model().to(device)
        m.load_state_dict(torch.load(p, map_location=device)['state'])
        m.eval()
        models.append(m)
        print(f'Loaded {p}')

print(f'Ensemble size: {len(models)}')

@torch.no_grad()
def ensemble_probs(models, loader, device):
    all_probs, all_masks = [], []
    for b in loader:
        img = b['image'].to(device)
        probs = torch.stack([
            torch.sigmoid(m(img).float())
            for m in models
        ]).mean(0).cpu()
        all_probs.append(probs)
        all_masks.append(b['mask'])
    return torch.cat(all_probs), torch.cat(all_masks)

tp, tm = ensemble_probs(models, test_loader, device)

print(f'\n{"Thr":>5} {"IoU":>7} {"F1":>7} {"FP":>12}')
for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]:
    r = metrics(tp, tm, t)
    print(f'{t:.2f}  {r["iou"]:.4f}  {r["f1"]:.4f}  {r["FP"]:>12,}')

Loaded /content/drive/MyDrive/galaxeye_checkpoints/ep07.pth
Loaded /content/drive/MyDrive/galaxeye_checkpoints/ep10.pth
Loaded /content/drive/MyDrive/galaxeye_checkpoints/ep19.pth
Ensemble size: 3

  Thr     IoU      F1           FP
0.50  0.0194  0.0380     2,161,336
0.55  0.0232  0.0454     1,239,783
0.60  0.0227  0.0444       816,406
0.65  0.0221  0.0433       530,397
0.70  0.0204  0.0400       324,859
0.75  0.0182  0.0357       177,888
